# 03 — Concept Leakage / Sufficiency  ·  Build step 04 (pre-G3) · Gate-G3 input · Novelty I2

Estimates **I(y ; features | concepts)** — the extra predictive information the encoder carries beyond the
clinical concepts — with honest cross-validated held-out log-likelihood. **Low** ⇒ concepts sufficient / faithful
(Path A). **High** ⇒ the model needs info outside the clinical concepts (Path B; the collapse/leakage is the finding).

Needs `concepts_all.npz` (nb01) and `M2_features.npy` (step 02). Output: `leakage_report.json`.

In [ ]:
import sys, os
OWMTL_PKG = ".."
sys.path.insert(0, OWMTL_PKG)
import numpy as np, json
from collections import defaultdict
from owmtl.icbhi_data import KNOWN_DISEASES
OUT_DIR = "/kaggle/working"
Z = np.load(os.path.join(OUT_DIR, "concepts_all.npz"), allow_pickle=True)
X = Z["X"].astype("float32"); patient = Z["patient"]; split = Z["split"]
diagnosis = Z["diagnosis"]; concept_names = list(Z["concept_names"])
try:
    F = np.load(os.path.join(OUT_DIR, "M2_features.npy")).astype("float32")
    HAVE_FEATS = True
except Exception:
    F = np.zeros((len(X), 1), "float32"); HAVE_FEATS = False; print("NOTE: M2_features.npy missing")
# patient-level aggregation (known-class disease task)
lab_map = {d: i for i, d in enumerate(KNOWN_DISEASES)}
by = defaultdict(list)
for i in range(len(X)):
    if diagnosis[i] in lab_map: by[patient[i]].append(i)
pids, Xp, Fp, yp, spp = [], [], [], [], []
for pid, idxs in by.items():
    idxs = np.array(idxs); pids.append(pid)
    Xp.append(X[idxs].mean(0)); Fp.append(F[idxs].mean(0))
    yp.append(lab_map[diagnosis[idxs[0]]]); spp.append(split[idxs[0]])
pids, Xp, Fp, yp, spp = map(np.array, (pids, np.stack(Xp), np.stack(Fp), yp, spp))
print("patients", len(pids), "| classes", KNOWN_DISEASES)

In [ ]:
from owmtl.leakage import estimate_leakage
if not HAVE_FEATS:
    raise SystemExit("Leakage needs M2 features (step 02). Export M2_features.npy first.")
rep = estimate_leakage(yp, Xp, Fp, n_splits=5, seed=42)
print(json.dumps(rep, indent=2))
with open(os.path.join(OUT_DIR, "leakage_report.json"), "w") as fh: json.dump(rep, fh, indent=2)

**Read `leakage_bits` / `interpretation`.** This is a Gate-G3 input: combine it with the accuracy tradeoff
(nb02) and the intervention effect (nb04) to decide Path A vs Path B. Do **not** pick the path here — that is the
G3 decision, made once all three numbers exist.